<a href="https://colab.research.google.com/github/ErasmoR/Erasmor/blob/master/Proyecto_integrador3_Prueba_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip -q install pyreadstat itables scikit-learn statsmodels linearmodels

import os, json
import numpy as np
import pandas as pd
import pyreadstat

import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Probit
from scipy.stats import norm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 3.4 MB/s eta 0:00:00


In [14]:
CONFIG = {
    # Población de estudio
    "edad_min": 18,
    "edad_max": 65,

    # Identificadores de hogar (como en tu notebook)
    "id_hogar_cols": ["llave_sec", "provincia", "prov", "unidad", "cuest", "hogar"],

    # Nombres de columnas base (según tu dataset actual)
    "cols": {
        "edad": "Edad",
        "sexo": "Sexo",
        "grado": "Grado_alcanzado",
        "ocup_estado": "ocu_des",
        "sal_emp": "Salario_efectivo_empleado",
        "sal_ind": "Salario_efectivo_independiente",
        "internet": "Uso_internet_últimos_6_meses",
        "jefe": "Jefe_de_hogar",
        "asiste": "Asiste_a_la_escuela",
        "jubilacion": "Recibe_jubilacion_pension",
        "provincia": "provincia",
    },

    # Targets operacionales
    "targets": {
        # Inserción laboral (ocupado vs no ocupado)
        "insercion_laboral": {
            "type": "binary",
            "rule": "ocu_des == 'Ocupados' (case-insensitive)"
        }
    },

    # Etapas del estudio
    "stages": {
        "mincer": {
            "y": "log_ingreso",
            "X": ["educ_years", "experiencia", "experiencia2", "Sexo"]
        },
        "heckman": {
            "selection_y": "participa",
            "selection_X": ["Edad", "Sexo", "Jefe_de_hogar", "ingreso_per_capita", "dependencia", "provincia"],
            "outcome_y": "log_ingreso",
            "outcome_X": ["educ_years", "experiencia", "experiencia2", "Sexo", "IMR"]
        },
        "ml": {
            "y": "insercion_laboral"
        }
    }
}
CONFIG


{'edad_min': 18,
 'edad_max': 65,
 'id_hogar_cols': ['llave_sec',
  'provincia',
  'prov',
  'unidad',
  'cuest',
  'hogar'],
 'cols': {'edad': 'Edad',
  'sexo': 'Sexo',
  'grado': 'Grado_alcanzado',
  'ocup_estado': 'ocu_des',
  'sal_emp': 'Salario_efectivo_empleado',
  'sal_ind': 'Salario_efectivo_independiente',
  'internet': 'Uso_internet_últimos_6_meses',
  'jefe': 'Jefe_de_hogar',
  'asiste': 'Asiste_a_la_escuela',
  'jubilacion': 'Recibe_jubilacion_pension',
  'provincia': 'provincia'},
 'targets': {'insercion_laboral': {'type': 'binary',
   'rule': "ocu_des == 'Ocupados' (case-insensitive)"}},
 'stages': {'mincer': {'y': 'log_ingreso',
   'X': ['educ_years', 'experiencia', 'experiencia2', 'Sexo']},
  'heckman': {'selection_y': 'participa',
   'selection_X': ['Edad',
    'Sexo',
    'Jefe_de_hogar',
    'ingreso_per_capita',
    'dependencia',
    'provincia'],
   'outcome_y': 'log_ingreso',
   'outcome_X': ['educ_years', 'experiencia', 'experiencia2', 'Sexo', 'IMR']},
  'ml': {

Codigo 3


In [15]:
# === Celda 3: Subir .sav manualmente (Opción A) y cargarlo ===
from google.colab import files

uploaded = files.upload()               # botón para subir
sav_name = next(iter(uploaded.keys()))  # nombre real del archivo subido
print("✅ Archivo subido:", sav_name)

def load_sav_files(paths, labels=None):
    frames = []
    if labels is None:
        labels = [f"p{i+1}" for i in range(len(paths))]
    assert len(paths) == len(labels)

    for p, lab in zip(paths, labels):
        if not os.path.exists(p):
            raise FileNotFoundError(f"No existe en /content: {p}")
        df, meta = pyreadstat.read_sav(p)
        df["periodo"] = lab
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

SAV_PATHS = [sav_name]
PERIODOS  = ["2024"]

raw = load_sav_files(SAV_PATHS, PERIODOS)
print("raw shape:", raw.shape)
raw.head(3)

StopIteration: 

In [33]:
def enforce_population(df, edad_col, edad_min, edad_max):
    before = len(df)
    out = df.copy()
    out = out[(out[edad_col].notna()) & (out[edad_col] >= edad_min) & (out[edad_col] <= edad_max)]
    after = len(out)
    print(f"Población aplicada {edad_min}-{edad_max}: antes={before} después={after}")
    assert out[edad_col].min() >= edad_min and out[edad_col].max() <= edad_max
    return out

def build_household_id(df, id_cols):
    tmp = df[id_cols].astype(str)
    return tmp.apply(lambda r: "_".join(r.values), axis=1)

def add_household_features(df, id_cols, edad_col, sal_emp_col, sal_ind_col):
    out = df.copy()

    out["nino"] = (out[edad_col] < 15).astype(int)
    out["adulto_mayor"] = (out[edad_col] >= 65).astype(int)

    out[sal_emp_col] = out[sal_emp_col].fillna(0)
    out[sal_ind_col] = out[sal_ind_col].fillna(0)
    out["ingreso_total_individual"] = out[sal_emp_col] + out[sal_ind_col]

    hogar = out.groupby(id_cols).agg(
        ingreso_hogar=("ingreso_total_individual", "sum"),
        personas=(edad_col, "count"),
        ninos=("nino", "sum"),
        adultos_mayores=("adulto_mayor", "sum")
    ).reset_index()

    hogar["dependencia"] = (hogar["ninos"] + hogar["adultos_mayores"]) / hogar["personas"].replace(0, np.nan)
    hogar["ingreso_per_capita"] = hogar["ingreso_hogar"] / hogar["personas"].replace(0, np.nan)

    return out.merge(hogar, on=id_cols, how="left")

# Educación: opción A (si existe) o fallback a mapeo
EDUC_MAP = {
    # ⚠️ IMPORTANTE: Ajusta este mapa si NO tienes Nivel_academico_continuo.
    # Ejemplo (NO definitivo): 36:12, 53:16, ...
}

def compute_educ_years(df, grado_col):
    if "Nivel_academico_continuo" in df.columns:
        return pd.to_numeric(df["Nivel_academico_continuo"], errors="coerce")
    g = pd.to_numeric(df[grado_col], errors="coerce")
    return g.map(EDUC_MAP)

def make_targets(df, ocup_col):
    out = df.copy()
    out["insercion_laboral"] = (out[ocup_col].astype(str).str.strip().str.lower() == "ocupados").astype(int)
    return out

def add_mincer_features(df, edad_col):
    out = df.copy()
    out["educ_years"] = compute_educ_years(out, CONFIG["cols"]["grado"])

    # Experiencia potencial (ajusta el 6 si tu marco usa otra constante)
    out["experiencia"] = (out[edad_col] - out["educ_years"] - 6).clip(lower=0)
    out["experiencia2"] = out["experiencia"] ** 2

    # ingreso total + log (evita log(0))
    out["ingreso_total"] = out[CONFIG["cols"]["sal_emp"]].fillna(0) + out[CONFIG["cols"]["sal_ind"]].fillna(0)
    out["log_ingreso"] = np.log(out["ingreso_total"].replace(0, np.nan))
    return out

In [27]:
Edad = CONFIG["cols"]["edad"]

df = enforce_population(raw, Edad, CONFIG["edad_min"], CONFIG["edad_max"])
df = add_household_features(df, CONFIG["id_hogar_cols"], Edad, CONFIG["cols"]["sal_emp"], CONFIG["cols"]["sal_ind"])
df = make_targets(df, CONFIG["cols"]["ocup_estado"])
df = add_mincer_features(df, Edad)

df["hogar_id"] = build_household_id(df, CONFIG["id_hogar_cols"])

print("df shape:", df.shape)
df[["hogar_id", Edad, "educ_years", "experiencia", "insercion_laboral", "ingreso_total"]].head()

Población aplicada 18-65: antes=42925 después=24669
df shape: (24669, 224)


,hogar_id,Edad,educ_years,experiencia,insercion_laboral,ingreso_total
0,1.0_01_01_001_01_1,44.0,23.0,15.0,1,1516.0
1,1.0_01_01_001_01_1,21.0,18.0,0.0,1,600.0
2,2.0_01_01_001_02_1,22.0,18.0,0.0,1,700.0
3,2.0_01_01_001_02_1,26.0,18.0,2.0,1,850.0
4,3.0_01_01_001_03_1,34.0,18.0,10.0,1,800.0


In [19]:
DF_POP   = df.copy()                      # población 18–65 (todos) -> ML + descriptivos
DF_WAGE  = df[df["log_ingreso"].notna()]  # solo con ingreso válido -> Mincer
DF_LABOR = df.copy()                      # para Heckman

# participación (proxy): inserción laboral (puedes cambiar si tienes una variable PEA)
DF_LABOR["participa"] = DF_LABOR["insercion_laboral"].astype(int)

print("DF_POP", DF_POP.shape)
print("DF_WAGE", DF_WAGE.shape)
print("DF_LABOR", DF_LABOR.shape)

DF_POP (24669, 224)
DF_WAGE (16341, 224)
DF_LABOR (24669, 225)


In [28]:
def eda_tables(df, vars_list, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    # % nulos
    nulls = (df[vars_list].isna().mean() * 100).sort_values(ascending=False)
    nulls.to_csv(os.path.join(out_dir, "eda_nulos_pct.csv"))

    # describe num
    num_cols = [c for c in vars_list if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if num_cols:
        df[num_cols].describe().to_csv(os.path.join(out_dir, "eda_describe_num.csv"))

    # distribuciones cat
    cat_cols = [c for c in vars_list if c in df.columns and not pd.api.types.is_numeric_dtype(df[c])]
    for c in cat_cols:
        vc  = df[c].astype(str).value_counts(dropna=False)
        pct = df[c].astype(str).value_counts(normalize=True, dropna=False) * 100
        out = pd.concat([vc, pct], axis=1)
        out.columns = ["frecuencia", "porcentaje"]
        out.to_csv(os.path.join(out_dir, f"eda_dist_{c}.csv"))

    # cruces con target
    if "insercion_laboral" in df.columns:
        for c in cat_cols:
            ct = pd.crosstab(df[c].astype(str), df["insercion_laboral"], normalize="index")
            ct.to_csv(os.path.join(out_dir, f"eda_crosstab_{c}_vs_target.csv"))

EDA_VARS = [
    CONFIG["cols"]["sexo"], CONFIG["cols"]["grado"], "educ_years",
    CONFIG["cols"]["internet"], CONFIG["cols"]["provincia"],
    "ingreso_per_capita", "dependencia", "insercion_laboral"
]

eda_tables(DF_POP, EDA_VARS, os.path.join(OUT_DIR, "eda"))
print("EDA exportada en outputs/eda/")

EDA exportada en outputs/eda/


In [29]:
def run_mincer(df):
    stage = CONFIG["stages"]["mincer"]
    y = df[stage["y"]]

    X_cols = [c for c in stage["X"] if c in df.columns]
    X = df[X_cols].copy()
    X = pd.get_dummies(X, drop_first=True)
    X = sm.add_constant(X, has_constant="add")

    res = sm.OLS(y, X, missing="drop").fit(cov_type="HC3")  # robust SE
    return res

mincer_res = run_mincer(DF_WAGE)
print(mincer_res.summary())

coef = pd.DataFrame({"coef": mincer_res.params, "pvalue": mincer_res.pvalues})
coef.to_csv(os.path.join(OUT_DIR, "mincer_coef.csv"))
print("Export: outputs/mincer_coef.csv")

                            OLS Regression Results                            
Dep. Variable:            log_ingreso   R-squared:                       0.340
Model:                            OLS   Adj. R-squared:                  0.340
Method:                 Least Squares   F-statistic:                     1959.
Date:                Thu, 21 May 2026   Prob (F-statistic):               0.00
Time:                        04:32:37   Log-Likelihood:                -22064.
No. Observations:               16341   AIC:                         4.414e+04
Df Residuals:                   16336   BIC:                         4.418e+04
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            4.6968      0.035    133.001   

In [30]:
def run_heckman_two_step(df):
    stage = CONFIG["stages"]["heckman"]

    sel_y = df[stage["selection_y"]]                       # igual
    sel_y = pd.to_numeric(sel_y, errors="coerce")          # <-- NUEVO (evita dtype object)

    sel_X_cols = [c for c in stage["selection_X"] if c in df.columns]
    sel_X = df[sel_X_cols].copy()
    sel_X = pd.get_dummies(sel_X, drop_first=True)         # igual
    sel_X = sel_X.apply(pd.to_numeric, errors="coerce")    # <-- NUEVO (evita dtype object)

    # <-- NUEVO: limpiar inf/NaN y ALINEAR X e y antes del Probit
    data_sel = pd.concat([sel_y.rename("sel_y"), sel_X], axis=1)
    data_sel = data_sel.replace([np.inf, -np.inf], np.nan).dropna()
    sel_y = data_sel["sel_y"]
    sel_X = data_sel.drop(columns=["sel_y"])

    sel_X = sm.add_constant(sel_X, has_constant="add")     # igual (pero ahora sobre data limpia)

    probit = Probit(sel_y, sel_X).fit(disp=False)          # <-- CAMBIO mínimo: sin missing="drop"

    xb  = probit.predict(linear=True)
    pdf = norm.pdf(xb)
    cdf = norm.cdf(xb)
    imr = (pdf / np.clip(cdf, 1e-9, None))

    df2 = df.copy()
    df2["IMR"] = np.nan                                    # <-- NUEVO (para no desalinear)
    df2.loc[data_sel.index, "IMR"] = imr                   # <-- NUEVO (IMR solo filas usadas)

    df_out = df2[(df2[stage["selection_y"]] == 1) & (df2[stage["outcome_y"]].notna())].copy()
    out_y = df_out[stage["outcome_y"]]
    out_y = pd.to_numeric(out_y, errors="coerce")          # <-- NUEVO

    out_X_cols = [c for c in stage["outcome_X"] if c in df_out.columns]
    out_X = df_out[out_X_cols].copy()
    out_X = pd.get_dummies(out_X, drop_first=True)
    out_X = out_X.apply(pd.to_numeric, errors="coerce")    # <-- NUEVO

    # <-- NUEVO: limpiar/alinéar outcome también
    data_out = pd.concat([out_y.rename("out_y"), out_X], axis=1)
    data_out = data_out.replace([np.inf, -np.inf], np.nan).dropna()
    out_y = data_out["out_y"]
    out_X = data_out.drop(columns=["out_y"])



In [31]:
def infer_feature_types(X):
    num, cat = [], []
    for c in X.columns:
        if pd.api.types.is_numeric_dtype(X[c]):
            num.append(c)
        else:
            cat.append(c)
    return num, cat

def build_ml_dataset(df):
    y = df[CONFIG["stages"]["ml"]["y"]]

    features = [
        CONFIG["cols"]["edad"], CONFIG["cols"]["sexo"], CONFIG["cols"]["grado"],
        "educ_years", "experiencia",
        "ingreso_per_capita", "dependencia",
        CONFIG["cols"]["internet"], CONFIG["cols"]["provincia"], CONFIG["cols"]["jefe"]
    ]
    features = [c for c in features if c in df.columns]
    X = df[features].copy()
    groups = df["hogar_id"].copy()
    return X, y, groups

X, y, groups = build_ml_dataset(DF_POP)
num_cols, cat_cols = infer_feature_types(X)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

models = {
    "logreg": LogisticRegression(max_iter=2000),
    "rf": RandomForestClassifier(n_estimators=300, random_state=42),
    "gb": GradientBoostingClassifier(random_state=42)
}

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

ml_results = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", clf)])
    pipe.fit(X_train, y_train)

    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, proba)
    cm  = confusion_matrix(y_test, pred)

    ml_results[name] = {"auc": float(auc), "cm": cm.tolist()}

    print(f"\n=== {name} ===")
    print("AUC:", auc)
    print("Confusion matrix:\n", cm)
    print(classification_report(y_test, pred))

with open(os.path.join(OUT_DIR, "ml_results.json"), "w", encoding="utf-8") as f:
    json.dump(ml_results, f, ensure_ascii=False, indent=2)

print("Export: outputs/ml_results.json")


=== logreg ===
AUC: 0.819618100743901
Confusion matrix:
 [[ 613  754]
 [ 346 3243]]
              precision    recall  f1-score   support

           0       0.64      0.45      0.53      1367
           1       0.81      0.90      0.85      3589

    accuracy                           0.78      4956
   macro avg       0.73      0.68      0.69      4956
weighted avg       0.76      0.78      0.76      4956


=== rf ===
AUC: 0.8397026352365382
Confusion matrix:
 [[ 750  617]
 [ 418 3171]]
              precision    recall  f1-score   support

           0       0.64      0.55      0.59      1367
           1       0.84      0.88      0.86      3589

    accuracy                           0.79      4956
   macro avg       0.74      0.72      0.73      4956
weighted avg       0.78      0.79      0.79      4956


=== gb ===
AUC: 0.858613441909696
Confusion matrix:
 [[ 770  597]
 [ 363 3226]]
              precision    recall  f1-score   support

           0       0.68      0.56      0.62

In [32]:
rows = []

# Mincer
rows.append({"etapa":"mincer", "rol":"Y", "variable": CONFIG["stages"]["mincer"]["y"]})
for v in CONFIG["stages"]["mincer"]["X"]:
    rows.append({"etapa":"mincer", "rol":"X", "variable": v})

# Heckman selección
rows.append({"etapa":"heckman_selection", "rol":"Y", "variable": CONFIG["stages"]["heckman"]["selection_y"]})
for v in CONFIG["stages"]["heckman"]["selection_X"]:
    rows.append({"etapa":"heckman_selection", "rol":"X", "variable": v})

# Heckman outcome
rows.append({"etapa":"heckman_outcome", "rol":"Y", "variable": CONFIG["stages"]["heckman"]["outcome_y"]})
for v in CONFIG["stages"]["heckman"]["outcome_X"]:
    rows.append({"etapa":"heckman_outcome", "rol":"X", "variable": v})

# ML
rows.append({"etapa":"ml", "rol":"Y", "variable": CONFIG["stages"]["ml"]["y"]})

tabla = pd.DataFrame(rows)
tabla.to_csv(os.path.join(OUT_DIR, "tabla_variables_por_etapa.csv"), index=False)
tabla

,etapa,rol,variable
0,mincer,Y,log_ingreso
1,mincer,X,educ_years
2,mincer,X,experiencia
3,mincer,X,experiencia2
4,mincer,X,Sexo
5,heckman_selection,Y,participa
6,heckman_selection,X,Edad
7,heckman_selection,X,Sexo
8,heckman_selection,X,Jefe_de_hogar
9,heckman_selection,X,ingreso_per_capita
